# How an LLM works: from predicting the next word to chat
(Part 1 of 3)

ChatGPT and Claude feel like you are talking to something. In this lecture we take a small language model, run it on a laptop and watch what it does, one step at a time.

We will not look inside the neural network and there is no math. We treat the model as a function and look at what goes in and what comes out. That turns out to be enough to explain most of what you see in a chat window.

The model we use, Qwen3 0.6B, is tiny compared to the models behind commercial products. It runs on a laptop without a GPU and it is often wrong. That is fine: we are here for the mechanism, and the mechanism is the same.

We use two versions of it. Each has a page describing what it is and how it was made (a "model card"), worth a look: [Qwen3-0.6B-Base](https://huggingface.co/Qwen/Qwen3-0.6B-Base) and [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B).

In [ ]:
#!pip install torch "transformers>=4.51" pandas matplotlib

In [ ]:
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

torch.set_grad_enabled(False)               # we only run the model, we never train it
transformers.logging.set_verbosity_error()  # hide the library's warnings

#### Load a language model (the first run downloads roughly 2 GB)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base")
model     = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B-Base")

## LLMs work with tokens, not words

The model never sees letters. Text is first cut into pieces called **tokens**, and each token is replaced by a number.

In [ ]:
headline = "Apple shares rose 3.2% to $173.20 after the Federal Reserve held interest rates steady."

In [ ]:
ids = tokenizer.encode(headline)
ids

#### Which piece of text does each number stand for?

In [ ]:
[tokenizer.decode(i) for i in ids]

Notice:
- common words are one token each, and the space in front of a word is part of the token
- the price is cut into single digits. The model does not see the number 173.20, it sees `1`, `7`, `3`, `.`, `2`, `0`. Remember this when we ask a model to do arithmetic in Part 2.

In [ ]:
len(headline.split()), len(ids)   # words, tokens

Check out this great (and fun!) site: https://tiktokenizer.vercel.app/

#### Going back from numbers to text loses nothing

In [ ]:
tokenizer.decode(ids)

#### Words the tokenizer knows well get one token, the rest are spelled out in pieces

In [ ]:
[tokenizer.decode(i) for i in tokenizer.encode(" Microsoft Nvidia NVDA unhedged")]

Every model family cuts text up in its own way. To see how other models do it, paste some text into [Tiktokenizer](https://tiktokenizer.vercel.app).

## Tokens in, a score for every possible next token out

In [ ]:
prompt = "The central bank of the United States is called the Federal"
ids = tokenizer.encode(prompt, return_tensors="pt")   # "pt": give me a PyTorch tensor instead of a list
ids

#### Call the model. It is a function: token numbers go in...

In [ ]:
output = model(ids, logits_to_keep=1) # if we didn't specify logits_to_keep, the model would return logits for each input token, rather than just the last one (useful in training, not inference)

#### ...and scores come out

In [ ]:
output.logits.shape # batch size 1, tokens with outputs (see note above), output vocab 151,936

The last number is the size of the model's vocabulary: every token it could ever write. For each of the tokens we sent in, we get one score for *every* token in that vocabulary. The last row is the one we care about: the model's opinion about what comes **next**.

In [ ]:
scores = output.logits[0, -1] # 0 picks the first (and only) batch, -1 picks the last (and only) token's logits
scores

#### Turn the scores into probabilities (they now add up to 1)

In [ ]:
probs = torch.softmax(scores, dim=-1)
probs.sum()

#### Which tokens does the model think are most likely to come next?

In [ ]:
top = torch.topk(probs, 10) # 
pd.Series(top.values.tolist(), index=[tokenizer.decode(i) for i in top.indices])

#### Here the model was nearly certain. What about a sentence that could continue in many ways?

In [ ]:
prompt = "Stocks fell sharply on Monday as investors"
ids    = tokenizer.encode(prompt, return_tensors="pt")
probs  = torch.softmax(model(ids).logits[0, -1], dim=-1)
top    = torch.topk(probs, 10)

pd.Series(top.values.tolist(), index=[tokenizer.decode(i) for i in top.indices]).plot.barh()

**This is everything the model does: tokens in, a probability for each possible next token out.**

It does not produce sentences, paragraphs or answers. Everything else in this lecture is built around this one function.

## Pick a token, add it to the text, ask again

#### Take the most likely next token
Note that actual transnformers don't simply pick the most likely next token; they often pick the next item based on the probability assigned to it. See calls to `multinomial` and references to `temperature` later in lecture

In [ ]:
next_id = probs.argmax()
tokenizer.decode(next_id)

#### Glue it to the end of what we had

In [ ]:
ids = torch.cat([ids, next_id.reshape(1, 1)], dim=1) # append the predicted next token to the input sequence
tokenizer.decode(ids[0]) # using zero index to decode the first (and only) sequence in the batch

In [ ]:
probs   = torch.softmax(model(ids).logits[0, -1], dim=-1) # dim=-1 applies softmax over the vocabulary dimension
next_id = probs.argmax()
print(f"{tokenizer.decode(ids[0])} | {tokenizer.decode(next_id)}") # the sentence so far, then the token the model picks next
ids     = torch.cat([ids, next_id.reshape(1, 1)], dim=1)

Run the cell above a few more times. Each run adds exactly one token.

#### That is a loop

In [ ]:
for step in range(40):
    probs   = torch.softmax(model(ids).logits[0, -1], dim=-1)
    next_id = probs.argmax()
    ids     = torch.cat([ids, next_id.reshape(1, 1)], dim=1)
    print(tokenizer.decode(next_id), end="|")

Every piece between two `|` is one full trip through the model. When text appears a few words at a time in a chat window, that is not an animation: this loop is running.

Did an index level or a price show up in your output? Nobody looked it up. The model produced digits that are likely to follow that text.

#### The library has this loop built in. It is called `generate`

In [ ]:
streamer = TextStreamer(tokenizer, skip_prompt=True)   # print each new token as soon as it is picked

ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=40, do_sample=False, streamer=streamer);

## Why does the same question get different answers?

So far we always took the single most likely token, so we get the same text every time. Chat products do not do that. They roll dice, weighted by the probabilities.

#### Roll the dice for one token. Run this cell several times

In [ ]:
probs = torch.softmax(model(ids).logits[0, -1], dim=-1)
print(prompt, "|", tokenizer.decode(torch.multinomial(probs, 1)))

#### `do_sample=True` rolls the dice at every step of the loop

In [ ]:
for attempt in range(3):
    model.generate(ids, max_new_tokens=25, do_sample=True, streamer=streamer)
    print("-----")

#### "Temperature" sets how adventurous the dice are. Low: almost always a likely token
Internally, T is a parameter that scales the logits before applying the softmax: `softmax(logits / T)`.

- T < 1 sharpens the distribution, so the top tokens get even more of the probability.
- T = 1 leaves the model's probabilities unchanged.
- T > 1 flattens it, so unlikely tokens get picked more often. That's what the notebook's temperature=1.8 cell does, and it's why that output gets weird.
- T → 0 approaches greedy (always the top token). Exactly 0 would divide by zero.

In [ ]:
for attempt in range(3):
    model.generate(ids, max_new_tokens=25, do_sample=True, temperature=0.2, streamer=streamer)
    print("-----")

#### High: unlikely tokens get picked too

In [ ]:
for attempt in range(3):
    model.generate(ids, max_new_tokens=25, do_sample=True, temperature=1.8, streamer=streamer)
    print("-----")

#### In a terminal

Everything so far fits in a program of about ten lines: type the start of a text, watch the model continue it. Read the program first, it is short.

```
python 1_autocomplete.py
```

## A raw model does not answer questions, it continues text

Notice that our model is called Qwen/Qwen3-0.6B-Base. The word "Base" means that the model is not "instruction tuned" (explained later).

In [ ]:
question = "Why do bond prices fall when interest rates rise?"

ids = tokenizer.encode(question, return_tensors="pt")
model.generate(ids, max_new_tokens=60, do_sample=False, streamer=streamer);

No answer. The model was trained on a very large amount of text to do one thing: predict what comes next. In the text it learned from, what often follows a question like this one? More questions, as in a textbook exercise or an exam.

Nobody has told it that it is an assistant. A model at this stage is called a **base model**.

## Getting an answer: write the start of a conversation and let the model continue it

In [ ]:
transcript = """The following is a conversation between a user and a helpful, knowledgeable assistant.

User: Why do bond prices fall when interest rates rise?
Assistant:"""

In [ ]:
ids = tokenizer.encode(transcript, return_tensors="pt")
model.generate(ids, max_new_tokens=130, do_sample=False, streamer=streamer);

Two things happened:
- It answered. In a document that looks like this, the likely next text is an answer. <== Subtle trick!
- It did not stop there. It went on to write the **user's** next line, and then answered that too. The model is not the assistant. It is writing a script, and it will write every part in it.

#### So we stop the loop when the model starts to write the user's line

In [ ]:
model.generate(ids, max_new_tokens=130, do_sample=False, streamer=streamer,
               stop_strings=["User:"], tokenizer=tokenizer);

That is a chat product in miniature: a transcript, a model that continues it, and a rule for when to stop.

This is also how people used language models before chat models existed. You did not ask a question, you wrote the beginning of a document whose natural continuation was the thing you wanted, often with a few worked examples in it. The paper that made this famous is the GPT-3 paper, [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) (2020).

## Chat models: the same trick, trained in

Companies take a base model and train it further on a very large number of conversations, all written in one fixed format, with special tokens that mark who is speaking. The result is called an **instruct** or **chat** model. This is the kind of model you have been using all along.

The conversations used by the big companies are not public, but openly published sets look the same. Open one and read a few rows in the viewer:
- [UltraChat 200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k): multi-turn conversations
- [OpenAssistant](https://huggingface.co/datasets/OpenAssistant/oasst1): conversations written and rated by volunteers
- [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca) and [Dolly 15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k): single instructions with their answers

The paper behind the first ChatGPT models describes the idea: [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) (2022).

#### We are done with the base model, so free the memory it takes (about 2.4 GB)

In [ ]:
del model

#### Load the chat version of the same model (roughly 3 GB on the first run)

In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B") # <== Notice there is no "Base" in the model name
chat_model     = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
chat_streamer  = TextStreamer(chat_tokenizer, skip_prompt=True)

## The chat format, piece by piece

The conversations a chat model is trained on are all written the same way, so before we use the model we need to be able to read that format. A conversation is a sequence of **turns**, and every turn looks like this:

```
<|im_start|>ROLE
the text of the turn<|im_end|>
```

| Piece | Meaning |
|---|---|
| `<\|im_start\|>` | a turn starts here |
| the word right after it | the **role**: who is speaking in this turn |
| everything after that first line | what they said |
| `<\|im_end\|>` | the turn ends here |

There are three roles:

| Role | Who it is |
|---|---|
| `system` | the application. Instructions that hold for the whole conversation. Optional, and always first |
| `user` | the person typing |
| `assistant` | the model |

A short conversation, written out in full:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a bond?<|im_end|>
<|im_start|>assistant
A bond is a loan that investors make to a company or a government.<|im_end|>
<|im_start|>user
And a stock?<|im_end|>
<|im_start|>assistant
```

Look at the last line. The text we give the model always ends with an assistant turn that has been **started but not finished**. That is the same trick as ending our transcript with `Assistant:`. The model writes the rest of that turn, and closes it with `<|im_end|>` itself.

This particular format is called ChatML. Other model families use different markers for the same idea. The roles `system`, `user` and `assistant` are the same almost everywhere, and you will see them again in every API.

#### The markers are not ordinary text. Each one is a single token that exists only for this purpose

In [ ]:
[chat_tokenizer.decode(i) for i in chat_tokenizer.encode("<|im_start|>user\nWhat is a bond?<|im_end|>")]

#### Now write a real prompt in that format: one user turn, and an open assistant turn

In [ ]:
question = "In one sentence, why do bond prices fall when interest rates rise?"

prompt = f"""<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
print(prompt)

(The `<think>` lines are not part of the format above. Ignore them for now, we come back to them at the end.)

In [ ]:
ids = chat_tokenizer.encode(prompt, return_tensors="pt")
chat_model.generate(ids, max_new_tokens=150, do_sample=False, streamer=chat_streamer);

Look at the very end of the output: the model itself wrote `<|im_end|>`. It learned from its training conversations that a turn ends with that token.

That is how a chat model "knows" it is done. `generate` watches for that token and stops. Here is our own loop from earlier with the same rule added:

In [ ]:
end_id = chat_tokenizer.encode("<|im_end|>")[0]

for step in range(150):
    next_id = chat_model(ids).logits[0, -1].argmax()
    if next_id == end_id:
        break
    ids = torch.cat([ids, next_id.reshape(1, 1)], dim=1)
    print(chat_tokenizer.decode(next_id), end="")

#### What libraries and APIs do for you

Nobody types these markers by hand. Every provider's API takes a **list of messages**, each with a role, and turns it into text like ours behind the scenes. In the library we are using, that step is called `apply_chat_template`:

In [ ]:
messages = [
    {"role": "user", "content": question},
]
chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False) ##== prompt

In [ ]:
prompt

Exactly the text we wrote by hand. (`add_generation_prompt` adds the open assistant turn at the end, `enable_thinking=False` adds the `<think>` lines.)

The markers differ from one model family to the next, which is the reason this is left to a library: [chat templates](https://huggingface.co/docs/transformers/chat_templating). When you call a commercial model you send the list of messages and never see the text. It is still there.

## Chat memory: the whole conversation is sent again, every time

#### Tell the model something

In [ ]:
prompt = """<|im_start|>user
My name is Marge and I own 100 shares of Apple.<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
ids = chat_tokenizer.encode(prompt, return_tensors="pt")
chat_model.generate(ids, max_new_tokens=150, do_sample=False, streamer=chat_streamer);

#### Now ask about it in a new call

In [ ]:
prompt = """<|im_start|>user
How many shares do I own, and of which company?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
ids = chat_tokenizer.encode(prompt, return_tensors="pt")
chat_model.generate(ids, max_new_tokens=150, do_sample=False, streamer=chat_streamer);

It has no idea. The model is the same function as before: text in, next token out. Nothing from the first call is kept anywhere.

#### In a terminal

The same thing as a program: a chat where every question is sent on its own.

```
python 2_chat_without_memory.py
```

#### So a chat application keeps the text of the conversation, and sends all of it on every turn

Here is a whole chat application. `conversation` is one piece of text that keeps growing: we add the user's turn, the model continues it, and we add what the model wrote.

In [ ]:
questions = [
    "My name is Marge and I own 100 shares of Apple.",
    "How many shares do I own, and of which company?",
    "I just bought 50 more. How many do I own now?",
]

In [ ]:
conversation = ""

for question in questions:
    conversation += f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    ids = chat_tokenizer.encode(conversation, return_tensors="pt")
    print(f"----- sending {len(ids[0])} tokens -----")
    output = chat_model.generate(ids, max_new_tokens=150, do_sample=False, streamer=chat_streamer)
    conversation += chat_tokenizer.decode(output[0][len(ids[0]):]) + "\n"
    #             ^--------- Chat being appended here

#### This is everything the model was given for the last question, plus its reply

In [ ]:
print(conversation)

Now it "remembers". The model did not change, and it still keeps nothing between calls. The memory is the text.

Look at the token counts: every turn sends more than the one before. This is why long conversations get slower and cost more, and why every model has a limit on how much text it can take in at once (its **context window**).

One more thing. On every turn the model reads the conversation again from the very first token, even though most of it is the same text as last time. Providers avoid redoing that work: they keep what was computed for the unchanged beginning and only compute the new part. This is called **prompt caching**, and it is why API price lists have a separate, cheaper line for "cached" input tokens ([how one provider describes it](https://platform.claude.com/docs/en/build-with-claude/prompt-caching)).

#### Problem set

Turn this loop into a program you can chat with in a terminal. Start from `2_chat_without_memory.py`: it takes about four changed lines.

## System prompts are also just text

#### A system prompt is one more turn, placed in front of the conversation

In [ ]:
prompt = """<|im_start|>system
You work in compliance at a bank. Start every answer with 'This is not investment advice.'<|im_end|>
<|im_start|>user
In one sentence, why do bond prices fall when interest rates rise?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
ids = chat_tokenizer.encode(prompt, return_tensors="pt")
chat_model.generate(ids, max_new_tokens=150, do_sample=False, streamer=chat_streamer);

Chat products have a system prompt you do not see. It is how the company tells the model its name, today's date and how to behave. Some companies publish theirs, and they are long: [Claude's system prompts](https://platform.claude.com/docs/en/release-notes/system-prompts).

## "Thinking" is the model writing notes to itself

The model picks one token at a time and cannot go back. What happens if it must give the answer immediately?

#### We can force that, by writing the first words of the assistant's turn ourselves

In [ ]:
question = "A stock is at $100. It rises 10% and then falls 10%. What is the price now?"

prompt = f"""<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

</think>

The answer is"""
print(prompt)

In [ ]:
ids = chat_tokenizer.encode(prompt, return_tensors="pt")
chat_model.generate(ids, max_new_tokens=150, do_sample=False, streamer=chat_streamer);

Is the first number right? Now read the steps underneath it.

The model had to commit to a number in its very next tokens, with no room to work anything out. Only then did it write an explanation, and the explanation could not change what was already on the page. A model cannot take a token back, so the text that follows tends to agree with it.

Notice also what we just did: we put words in the assistant's mouth and the model carried on from them. To the model the assistant's turn is text like any other.

#### Now start its turn with different words

In [ ]:
prompt = f"""<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

</think>

Let's think step by step."""
print(prompt)

In [ ]:
ids = chat_tokenizer.encode(prompt, return_tensors="pt")
chat_model.generate(ids, max_new_tokens=300, do_sample=False, streamer=chat_streamer);

Same model, same question. This time it writes out the steps first, and each step it writes becomes part of the text the next tokens are predicted from. By the time it reaches the answer, the working is on the page in front of it.

This trick was one of the first things people discovered about these models, and for a year or two everybody who used them knew it:
- **Chain of thought**: show the model worked examples and it works step by step too. [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903) (2022)
- **"Let's think step by step"**: adding that one sentence is enough. [Large Language Models are Zero-Shot Reasoners](https://arxiv.org/abs/2205.11916) (2022)

#### Today it is trained in

Newer models are trained on conversations where the assistant first writes its working between `<think>` and `</think>`, and then the answer. That is what the lines we have been ignoring are for: by writing an **empty** `<think>` block ourselves, we told the model its thinking was already finished.

Leave the block out and the model writes it:

In [ ]:
prompt = f"""<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
print(prompt)

In [ ]:
ids = chat_tokenizer.encode(prompt, return_tensors="pt")   # this one takes a while
output = chat_model.generate(ids, max_new_tokens=1000, do_sample=False, streamer=chat_streamer)

There is no separate reasoning machinery. The notes are tokens from the same loop.

It is not free. Count the tokens it wrote for one small question:

In [ ]:
len(output[0]) - len(ids[0])

In libraries and APIs this is a switch: `enable_thinking` in the library we are using, "extended thinking" or "reasoning effort" in commercial products ([one provider's description](https://platform.claude.com/docs/en/build-with-claude/extended-thinking)). Chat products usually hide the notes or show a summary. You pay for those tokens either way.

## What to take away

- A language model does one thing: given tokens, it gives a probability for each possible next token.
- Text is produced one token at a time, by a loop around the model.
- The same prompt gives different answers because the loop rolls dice.
- A base model continues text. It answers only when the text it is continuing looks like a place where an answer comes next.
- A chat model is a base model trained on conversations in one fixed format. A chat is one long text with markers for who is speaking.
- The model remembers nothing between calls. The application sends the whole conversation every time.
- System prompts and thinking are more text in that same conversation.

In Part 2 we use the same ingredients to let a model *do* things: run code, query data, and keep going until a job is done.

## Things to try at home

- Put your own headline through the tokenizer. Try a ticker, a large number, a date, a word in another language.
- Change the prompt in the loop and watch which tokens the model is unsure about.
- Write your own transcript for the base model (run the cell that loads it again first, we deleted it). Change the first line to describe a rude assistant, or a pirate. What changes?
- In the chat loop, add a fourth question. Then edit `conversation` by hand before asking it, so that the history says something that never happened. What does the model do?
- Start the assistant's turn with words of your own, as we did with "The answer is".
- Ask the chat model something it cannot know, such as today's date or yesterday's closing price of a stock. What does it say?

## Attribution
Written with assistance from Claude Auto (including Opus 5.5)